# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HamzaMohamed25/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis (The Grain):

One row represents one unique content item (content_hash_id) over a specific historical performance snapshot.

Time Window:

We will use a mid-panel month (March 2026) as our observation window to build features, predicting the decay proxy (Label) in the subsequent 90 days. We deliberately avoid June 2026 as it is our blind test set.



In [3]:
import pandas as pd
import numpy as np
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

print("1. Loading dim_content directly...")
df_dim = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet",
    storage_options={"token": hf_token}
)

print("2. Loading ONLY March 2026 partition (Zero RAM overhead)...")
df_fact_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/",
    storage_options={"token": hf_token}
)

print("3. Aggregating to our target Grain (1 Row = 1 Content URL)...")
march_agg = df_fact_march.groupby('content_hash_id').agg({
    'gsc_clicks': 'sum',
    'gsc_impressions': 'sum',
    'gsc_avg_position': 'mean'
}).reset_index()

march_agg = march_agg.rename(columns={
    'gsc_clicks': 'clicks_past_30d',
    'gsc_impressions': 'impressions_past_30d',
    'gsc_avg_position': 'avg_position_past_30d'
})

print("4. Merging and finalizing Data Contract features...")
df = pd.merge(df_dim, march_agg, on='content_hash_id', how='inner')

df['target_proxy_decay'] = np.where(df['impressions_past_30d'] < df['impressions_past_30d'].median(), 1, 0)

print(f"\n--- SUCCESS ---")
print(f"Total Rows: {len(df)}")
print(f"Is one row = one unique content_hash_id? {df['content_hash_id'].is_unique}")

1. Loading dim_content directly...
2. Loading ONLY March 2026 partition (Zero RAM overhead)...
3. Aggregating to our target Grain (1 Row = 1 Content URL)...
4. Merging and finalizing Data Contract features...

--- SUCCESS ---
Total Rows: 331437
Is one row = one unique content_hash_id? True


## 2. Fields: feature / label / context / excluded

Features (Knowable at decision time):

impressions_past_30d, clicks_past_30d, avg_position_past_30d (Aggregated historical performance from Google Search Console).

Label (Proxy for prediction):

target_proxy_decay (1 if trend indicates decay, 0 otherwise). Used only during training to teach the model what decay looks like.

Context:

content_hash_id (Used for row identification and joins, must never be passed to the model).

Excluded:

Any future metrics (e.g., clicks in April). Excluded explicitly because it violates causality (Data Leakage). We cannot know future clicks when deciding to refresh a page in March.

In [4]:
features = ['impressions_past_30d', 'clicks_past_30d', 'avg_position_past_30d']
label = 'target_proxy_decay'
context = 'content_hash_id'
excluded = 'future_clicks'

print(f"Features (Knowable): {features}")
print(f"Label (Prediction Proxy): {label}")
print(f"Context (Must not train on): {context}")
print(f"Excluded (To prevent leakage): {excluded}")

Features (Knowable): ['impressions_past_30d', 'clicks_past_30d', 'avg_position_past_30d']
Label (Prediction Proxy): target_proxy_decay
Context (Must not train on): content_hash_id
Excluded (To prevent leakage): future_clicks


## 3. Verify it with queries (grain, counts, missing values, windows)

Here we prove our contract claims using queries:

Verifying the grain uniqueness.

Checking the availability of our features (missing values).

The Trap: Demonstrating that if we accidentally leak the label into the features, the data is invalid for decision-support.


In [5]:
print(f"Row count ({df.shape[0]}) matches unique IDs ({df['content_hash_id'].nunique()}).")

print("\nMissing values in features:")
print(df[features].isnull().sum())

df['LEAKED_future_data'] = df['target_proxy_decay']
print("\nFact 3: THE TRAP")
print("If we train using 'LEAKED_future_data', the model will cheat with 100% accuracy.")
df = df.drop(columns=['LEAKED_future_data'])
print("Trap neutralized!")

Row count (331437) matches unique IDs (331437).

Missing values in features:
impressions_past_30d          0
clicks_past_30d               0
avg_position_past_30d    154699
dtype: int64

Fact 3: THE TRAP
If we train using 'LEAKED_future_data', the model will cheat with 100% accuracy.
Trap neutralized!


## 4. Data limits

Limitations of this dataset:

No intent/market context: The data indicates that impressions dropped, but cannot tell us why (e.g., search volume for the topic dropped globally).

Anonymized Queries: Google Search Console hides data for low-volume queries. We may miss long-tail keyword decay.

Directional, not causal: The model provides decision-support by flagging candidates likely to decay. It does not guarantee that refreshing the content will reverse it.

In [6]:
# Data limits confirmed.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.